In [1]:
import torch
import transformers
import gradio
import pydantic

print('PyTorch', torch.__version__)
print('Transformers', transformers.__version__)
print('Gradio', gradio.__version__)
print('Pydantic', pydantic.__version__)

assert torch.cuda.is_available(), '目前沒偵測到 GPU，請切換到運行 GPU 的執行模式'
print('GPU', torch.cuda.get_device_name(0))

PyTorch 2.11.0+cu128
Transformers 5.15.0
Gradio 6.24.0
Pydantic 2.13.4
GPU Tesla T4


In [2]:
from huggingface_hub import login

HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("已使用 Colab Secret 登入 Hugging Face。")
else:
    print(
        "未設定 HF_TOKEN。若模型可以公開下載，仍可繼續；"
        "若出現 401 或 403，請設定 Hugging Face Token。"
    )

已使用 Colab Secret 登入 Hugging Face。


In [3]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "google/gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    padding_side="left",
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
    attn_implementation="sdpa",
)

model.eval()

print("模型載入完成：", MODEL_ID)
print("模型主要裝置：", model.device)

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

模型載入完成： google/gemma-4-E2B-it
模型主要裝置： cuda:0


In [4]:
for name, token in processor.tokenizer.special_tokens_map.items():
    token_id = processor.tokenizer.convert_tokens_to_ids(token)
    print(f'{name} -> {token} -> {token_id}')

bos_token -> <bos> -> 2
eos_token -> <eos> -> 1
unk_token -> <unk> -> 3
pad_token -> <pad> -> 0
mask_token -> <mask> -> 4
audio_token -> <|audio|> -> 258881
boa_token -> <|audio> -> 256000
boi_token -> <|image> -> 255999
eoa_token -> <audio|> -> 258883
eoc_token -> <channel|> -> 101
eoi_token -> <image|> -> 258882
eot_token -> <turn|> -> 106
escape_token -> <|"|> -> 52
etc_token -> <tool_call|> -> 49
etd_token -> <tool|> -> 47
etr_token -> <tool_response|> -> 51
image_token -> <|image|> -> 258880
soc_token -> <|channel> -> 100
sot_token -> <|turn> -> 105
stc_token -> <|tool_call> -> 48
std_token -> <|tool> -> 46
str_token -> <|tool_response> -> 50
think_token -> <|think|> -> 98


In [5]:
from typing import Literal, Any
from pydantic import BaseModel, Field


MAX_TOOL_ROUNDS = 3

# 先關閉 thinking，速度較快。
# 若模型經常沒有正確選擇工具，可以改成 True。
ENABLE_THINKING = False


SYSTEM_PROMPT = """
你是一個專業、高效率的繁體中文 AI 助理。系統已為你配備了外部查詢工具，以協助你取得即時資訊。

【工具呼叫總原則】
1. 意圖配對：請務必仔細閱讀系統所提供之工具的「描述 (description)」。當使用者的問題符合工具功能時，請優先調用該工具，絕對不可憑藉自身訓練記憶來回答需要即時數據的問題。
2. 參數嚴格防呆：呼叫工具前，請確認使用者已提供所有必填參數（例如：特定地點、名稱等）。若資訊不足，請務必先向使用者提問釐清，【嚴禁】自行猜測、隨機代入預設值或捏造參數。
3. 忠實轉述結果：收到工具回傳的資料後，請直接根據該真實數據進行統整，並以自然流暢的繁體中文回答使用者，不可捏造工具未提供的額外數值。
4. 處理無工具情境：若使用者的問題不屬於任何可用工具的範圍（例如：日常閒聊、一般名詞解釋、寫作建議），請直接以自然語言回覆，無需嘗試呼叫任何工具。
5. 錯誤回報：若工具執行失敗或無法取得資料，請誠實告知使用者系統目前無法查詢，不可假造答案。
6. 語言轉換：當使用者查詢某地點的天氣時，如果使用者給的地點資訊不是英文的話，請將該地點翻譯成英文再送給查詢工具，例如:「查詢台北的天氣」調用工具時要用 Taipei 進行查詢。
""".strip()


WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": (
            "查詢指定城市或地點目前的即時天氣。"
            "適用於目前溫度、體感溫度、濕度、降雨、雲量與風速。"
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": (
                        "城市或地點名稱，例如：Taipei、Tokyo、New York。"
                    ),
                },
                "unit": {
                    "type": "string",
                    "enum": [
                        "celsius",
                        "fahrenheit",
                    ],
                    "description": (
                        "溫度單位；未指定時使用 celsius。"
                    ),
                },
            },
            "required": ["location"],
        },
    },
}

TOOLS = [WEATHER_TOOL]


class WeatherArguments(BaseModel):
    """實際執行工具前，用來驗證模型產生的參數。"""

    location: str = Field(
        min_length=2,
        max_length=100,
    )

    unit: Literal[
        "celsius",
        "fahrenheit",
    ] = "celsius"

In [6]:
import requests


WMO_WEATHER_CODES = {
    0: "晴朗",
    1: "大致晴朗",
    2: "局部多雲",
    3: "陰天",
    45: "有霧",
    48: "霧淞",
    51: "輕微毛毛雨",
    53: "中等毛毛雨",
    55: "強烈毛毛雨",
    56: "輕微凍毛毛雨",
    57: "強烈凍毛毛雨",
    61: "小雨",
    63: "中雨",
    65: "大雨",
    66: "輕微凍雨",
    67: "強烈凍雨",
    71: "小雪",
    73: "中雪",
    75: "大雪",
    77: "雪粒",
    80: "輕微陣雨",
    81: "中等陣雨",
    82: "強烈陣雨",
    85: "輕微陣雪",
    86: "強烈陣雪",
    95: "雷雨",
    96: "雷雨伴隨小冰雹",
    99: "雷雨伴隨強冰雹",
}


HTTP = requests.Session()

HTTP.headers.update({
    "User-Agent": "gemma4-colab-tool-calling-demo/1.0"
})


def geocode_location(location: str) -> dict[str, Any]:
    """
    將地點名稱轉成經緯度。
    """

    response = HTTP.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={
            "name": location,
            "count": 1,
            "language": "zh",
            "format": "json",
        },
        timeout=20,
    )

    response.raise_for_status()

    payload = response.json()
    results = payload.get("results") or []

    if not results:
        raise ValueError(f"找不到地點：{location}")

    place = results[0]

    return {
        "name": place.get("name", location),
        "admin1": place.get("admin1"),
        "country": place.get("country"),
        "latitude": place["latitude"],
        "longitude": place["longitude"],
        "timezone": place.get("timezone"),
    }


def get_current_weather(
    location: str,
    unit: str = "celsius",
) -> dict[str, Any]:
    """
    查詢指定地點目前的天氣。

    Args:
        location: 城市或地點名稱。
        unit: celsius 或 fahrenheit。

    Returns:
        包含目前天氣資料的字典。
    """

    place = geocode_location(location)

    response = HTTP.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": (
                "temperature_2m,"
                "relative_humidity_2m,"
                "apparent_temperature,"
                "precipitation,"
                "weather_code,"
                "cloud_cover,"
                "wind_speed_10m"
            ),
            "temperature_unit": (
                "fahrenheit"
                if unit == "fahrenheit"
                else "celsius"
            ),
            "wind_speed_unit": "kmh",
            "timezone": "auto",
        },
        timeout=20,
    )

    response.raise_for_status()

    payload = response.json()
    current = payload.get("current")

    if not current:
        raise RuntimeError(
            "天氣 API 沒有回傳 current 資料。"
        )

    weather_code = current.get("weather_code")

    location_parts = [
        place.get("name"),
        place.get("admin1"),
        place.get("country"),
    ]

    return {
        "resolved_location": "，".join(
            str(part)
            for part in location_parts
            if part
        ),
        "latitude": place["latitude"],
        "longitude": place["longitude"],
        "timezone": (
            payload.get("timezone")
            or place.get("timezone")
        ),
        "observation_time": current.get("time"),
        "temperature": current.get("temperature_2m"),
        "temperature_unit": (
            payload
            .get("current_units", {})
            .get("temperature_2m")
        ),
        "apparent_temperature": (
            current.get("apparent_temperature")
        ),
        "relative_humidity_percent": (
            current.get("relative_humidity_2m")
        ),
        "precipitation_mm": (
            current.get("precipitation")
        ),
        "cloud_cover_percent": (
            current.get("cloud_cover")
        ),
        "wind_speed_kmh": (
            current.get("wind_speed_10m")
        ),
        "weather_code": weather_code,
        "weather_description": (
            WMO_WEATHER_CODES.get(
                weather_code,
                "未知天氣狀況",
            )
        ),
        "data_source": "Open-Meteo",
    }

In [7]:
test_get_weather = get_current_weather('Taipei')

test_get_weather

{'resolved_location': '台北市，臺灣省 or 台灣省，台湾',
 'latitude': 25.05306,
 'longitude': 121.52639,
 'timezone': 'Asia/Taipei',
 'observation_time': '2026-08-26T00:00',
 'temperature': 25.3,
 'temperature_unit': '°C',
 'apparent_temperature': 31.6,
 'relative_humidity_percent': 96,
 'precipitation_mm': 0.1,
 'cloud_cover_percent': 100,
 'wind_speed_kmh': 1.6,
 'weather_code': 51,
 'weather_description': '輕微毛毛雨',
 'data_source': 'Open-Meteo'}

In [8]:
TOOL_REGISTRY = {
    'get_current_weather':{
        'function': get_current_weather,
        'validator': WeatherArguments
    }
}

In [9]:
import json
import re

def cast_tool_value(value:str) -> Any:
    """
    把 tool call 中的文字轉成適當的 Python 型別
    """
    value = value.strip()

    if value.lower() == 'true':
        return True
    if value.lower() == 'false':
        return False
    if value.lower() == 'null':
        return None

    try:
        return int(value)
    except ValueError:
        pass

    try:
        return float(value)
    except ValueError:
        pass

    return value.strip("'\"")


def extract_tool_call_with_regex(text:str) -> list[dict[str, Any]]:
    """
    解析 Gemma 4 原始的 tool-call token 格式
    """
    calls = []
    matches = re.findall(
        r'<\|tool_call>call:(\w+)\{(.*?)\}<tool_call\|>',
        text,
        flags = re.DOTALL
    )

    for function_name, raw_args in matches:
        arguments = {}

        arguments_matches = re.findall(
            r'(\w+):(?:<\|"\|>(.*?)<\|"\|>|([^,}]*))',
            raw_args,
            flags = re.DOTALL
        )
        for key, quotated_value, plain_value in arguments_matches:
            raw_value = (
                quotated_value
                if quotated_value != ''
                else palin_value
            )

            arguments[key] = cast_tool_value(raw_value)

        calls.append(
            {
                'name':function_name,
                'arguments':arguments
            }
        )
    return calls

def normalize_parsed_tool_calls(parsed_calls: list[dict[str,Any]] | None) -> list[dict[str,Any]]:
    """
    將 processor.parse_response 的結果統一成:
    {
    'name':...,
    'arguments':{...}
    }
    """
    normalized = []

    for call in parsed_calls or []:
        function = call.get('function')

        name = function.get('name')
        arguments = function.get('arguments')

        if isinstance(arguments, str):
            try:
                arguments = json.loads(arguments)
            except JSONDecodeError:
                arguments = {}
        if name:
            normalized.append({
                'name': name,
                'arguments': arguments
            })
    return normalized


def clean_generated_text(text:str) -> str:
    """
    移除模型回答中的 special token
    """
    cleaned = text

    tokens_to_remove = (

        # 模型生成結束 token
        '<eos>',
        '<turn|>',

        # Function calling protocol
        '<|tool_call>',
        '<tool_call|>',
        '<|tool_response>',
        '<tool_response|>',

        # Thinking channel protocol
        '<|channel>',
        '<channel|>'

    )
    for token in tokens_to_remove:
        cleaned = cleaned.replace(token,'')

    cleaned = re.sub(
        r'call:\w\{.*?\}',
        '',
        cleaned
    )

    return cleaned.strip()

def parse_model_output(raw_output:str) -> dict[str, Any]:
    """
    將 Gemma 4 輸出解析成一般回答或 tool call
    """

    if hasattr(processor, 'parse_response'):
        try:
            parsed = processor.parse_response(raw_output)
            tool_calls = normalize_parsed_tool_calls(
                parsed.get('tool_calls')
            )

            content = (
                parsed.get('content') or ''
            ).strip()

            if tool_calls or content:
                return {
                    'content':content,
                    'tool_calls':tool_calls,
                    'thinking':parsed.get('thinking')
                }
        except Exception:
            pass

    tool_calls = extract_tool_call_with_regex(raw_output)

    return {
        'content':(
            ''
            if tool_calls
            else clean_generated_text(raw_output)
        ),
        'tool_calls': tool_calls,
        'thinking': None
    }

In [10]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": "新竹現在天氣如何？"
    }
]
prompt = processor.apply_chat_template(
        messages,
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )

print(f'prompt: {prompt}')

prompt: <bos><|turn>system
你是一個專業、高效率的繁體中文 AI 助理。系統已為你配備了外部查詢工具，以協助你取得即時資訊。

【工具呼叫總原則】
1. 意圖配對：請務必仔細閱讀系統所提供之工具的「描述 (description)」。當使用者的問題符合工具功能時，請優先調用該工具，絕對不可憑藉自身訓練記憶來回答需要即時數據的問題。
2. 參數嚴格防呆：呼叫工具前，請確認使用者已提供所有必填參數（例如：特定地點、名稱等）。若資訊不足，請務必先向使用者提問釐清，【嚴禁】自行猜測、隨機代入預設值或捏造參數。
3. 忠實轉述結果：收到工具回傳的資料後，請直接根據該真實數據進行統整，並以自然流暢的繁體中文回答使用者，不可捏造工具未提供的額外數值。
4. 處理無工具情境：若使用者的問題不屬於任何可用工具的範圍（例如：日常閒聊、一般名詞解釋、寫作建議），請直接以自然語言回覆，無需嘗試呼叫任何工具。
5. 錯誤回報：若工具執行失敗或無法取得資料，請誠實告知使用者系統目前無法查詢，不可假造答案。
6. 語言轉換：當使用者查詢某地點的天氣時，如果使用者給的地點資訊不是英文的話，請將該地點翻譯成英文再送給查詢工具，例如:「查詢台北的天氣」調用工具時要用 Taipei 進行查詢。<|tool>declaration:get_current_weather{description:<|"|>查詢指定城市或地點目前的即時天氣。適用於目前溫度、體感溫度、濕度、降雨、雲量與風速。<|"|>,parameters:{properties:{location:{description:<|"|>城市或地點名稱，例如：Taipei、Tokyo、New York。<|"|>,type:<|"|>STRING<|"|>},unit:{description:<|"|>溫度單位；未指定時使用 celsius。<|"|>,enum:[<|"|>celsius<|"|>,<|"|>fahrenheit<|"|>],type:<|"|>STRING<|"|>}},required:[<|"|>location<|"|>],type:<|"|>OBJECT<|"|>}}<tool|><turn|>
<|turn>user
新竹現在天氣如何？<turn|>
<|turn>model



In [11]:
inputs = processor(
        text=prompt,
        return_tensors="pt",
    ).to(model.device)

print(f'inputs: {inputs}')

inputs: {'input_ids': tensor([[     2,    105,   9731,    107, 237408,  90432, 109357, 236951, 237390,
          55894, 236918, 240200, 238952,  55041,  12498, 236743, 180314, 236924,
          53189, 237877, 237873, 237408, 237868, 238796, 237089,  49753, 165675,
          30398, 236900, 237162, 181143, 237408,  34893, 238270, 237479,  89998,
         236924,    108, 237604,  30398, 239156, 239138, 240018, 129912, 237598,
            107, 236770, 236761, 236743, 237588, 239387, 237868, 238403, 237184,
         239230, 238659, 238032, 240634, 238920, 111450,  53189, 237403,  12680,
         237437,  30398, 236918, 237374,  47710,    568,   7777, 236768,  51666,
         238834,   5938,  53877,  18053,  44684,  30398,  18478, 237479, 236900,
         239230, 118823, 238533, 237105, 239524,  30398, 236900, 171295,  25415,
         243466, 242536,  38137,  55616,  82324, 237967,  49695,  10042, 238270,
         237479,  90107, 123842, 236924,    107, 236778, 236761, 236743, 209458,
      

In [12]:
input_length = inputs["input_ids"].shape[-1]

outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        use_cache=True,
    )
generated_tokens = outputs[0][input_length:]

print(outputs)
print(generated_tokens)

tensor([[     2,    105,   9731,    107, 237408,  90432, 109357, 236951, 237390,
          55894, 236918, 240200, 238952,  55041,  12498, 236743, 180314, 236924,
          53189, 237877, 237873, 237408, 237868, 238796, 237089,  49753, 165675,
          30398, 236900, 237162, 181143, 237408,  34893, 238270, 237479,  89998,
         236924,    108, 237604,  30398, 239156, 239138, 240018, 129912, 237598,
            107, 236770, 236761, 236743, 237588, 239387, 237868, 238403, 237184,
         239230, 238659, 238032, 240634, 238920, 111450,  53189, 237403,  12680,
         237437,  30398, 236918, 237374,  47710,    568,   7777, 236768,  51666,
         238834,   5938,  53877,  18053,  44684,  30398,  18478, 237479, 236900,
         239230, 118823, 238533, 237105, 239524,  30398, 236900, 171295,  25415,
         243466, 242536,  38137,  55616,  82324, 237967,  49695,  10042, 238270,
         237479,  90107, 123842, 236924,    107, 236778, 236761, 236743, 209458,
         241656, 237829, 238

In [13]:
processor.decode(
        generated_tokens,
        skip_special_tokens=False,
    )

'<|tool_call>call:get_current_weather{location:<|"|>Hsinchu<|"|>}<tool_call|><|tool_response>'

In [14]:
import logging

logging.getLogger('transformers').setLevel(logging.ERROR)


@torch.inference_mode()
def generate_gemma(messages:list[dict[str,Any]], max_new_tokens:int=512) -> str:
    """
    將 messages 與 tools 套用 gemma 4 chat template, 並取得模型原始輸出。
    """
    prompt = processor.apply_chat_template(
        messages,
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING
    )

    inputs = processor(
        text = prompt,
        return_tensors='pt'
    ).to(model.device)

    input_length = inputs['input_ids'].shape[-1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        use_cache=True
    )

    generated_tokens = outputs[0][input_length:]

    return processor.decode(
        generated_tokens,
        skip_special_tokens=False
    )


In [15]:
generated_output = generate_gemma(messages=messages)
print(generated_output)

<|tool_call>call:get_current_weather{location:<|"|>Hsinchu<|"|>}<tool_call|><|tool_response>


In [16]:
from pydantic import ValidationError

def excute_tool_call(call:dict[str,Any]) -> dict[str,Any]:
    """
    驗證 function name, arguments 並執行允許的工具。
    """
    name = call.get('name')
    arguments = call.get('arguments',{})

    if name not in TOOL_REGISTRY:
        return {
            'name': name or 'unknown tool',
            'response': {
                'ok': False,
                'error': f'不允許或不存在的工具: {name}'
            }
        }

    tool = TOOL_REGISTRY[name]

    try:
        # Pydantic 參數驗證
        validated = tool['validator'].model_validate(arguments)

        # 執行 Python 函數
        result = tool['function'](**validated.model_dump())

        return {
            'name': name,
            'response':{
                'ok': True,
                'data':result
            }
        }
    except ValidationError as exc:
        return {
            'name': name,
            'response': {
                'ok': False,
                'error': '參數驗證未通過',
                'details': exc.errors(
                    include_url=False,
                    inclued_input=False
                )
            }
        }
    except requests.RequestException as exc:
        return {
            'name': name,
            'response': {
                'ok': False,
                'error': f'服務連線失敗:{exc}'
            }
        }
    except Exception as exc:
        return {
            'name': name,
            'response': {
                'ok': False,
                'error': str(exc)
            }
        }


In [17]:
parsed_output = parse_model_output(generated_output)
tool_calls = parsed_output['tool_calls']

print(parsed_output)
print(tool_calls)

for call in tool_calls:
    result = excute_tool_call(call)
    print(result)

{'content': '', 'tool_calls': [{'name': 'get_current_weather', 'arguments': {'location': 'Hsinchu'}}], 'thinking': None}
[{'name': 'get_current_weather', 'arguments': {'location': 'Hsinchu'}}]
{'name': 'get_current_weather', 'response': {'ok': True, 'data': {'resolved_location': '新竹市，臺灣省 or 台灣省，台湾', 'latitude': 24.80361, 'longitude': 120.96861, 'timezone': 'Asia/Taipei', 'observation_time': '2026-08-26T00:00', 'temperature': 25.1, 'temperature_unit': '°C', 'apparent_temperature': 31.3, 'relative_humidity_percent': 97, 'precipitation_mm': 0.1, 'cloud_cover_percent': 99, 'wind_speed_kmh': 1.8, 'weather_code': 51, 'weather_description': '輕微毛毛雨', 'data_source': 'Open-Meteo'}}}


In [18]:
import copy

def initial_messages() -> list[dict[str,Any]]:
    return [
        {
            'role': 'system',
            'content': SYSTEM_PROMPT
        }
    ]

def run_agent(
        user_message: str,
        conversation: list[dict[str,Any]] | None = None
        ) -> tuple[
            str,
            list[dict[str, Any]],
            list[dict[str, Any]]
            ]:
    """
    function calling orchestration

    Returns:
        answer:
            最終自然語言回答
        messages:
            包含 tool call 與 tool response 的完整 LLM 對話歷史
        trace:
            方便除錯與觀察的 agent 執行記錄
    """
    messages = copy.deepcopy(conversation) if conversation else initial_messages()

    messages.append(
        {
            'role': 'user',
            'content': user_message
        }
    )

    trace = []

    for round_index in range(MAX_TOOL_ROUNDS+1):
        raw_output = generate_gemma(messages = messages)
        parsed_output = parse_model_output(raw_output)
        calls = parsed_output['tool_calls']

        trace_item = {
            'round': round_index,
            'raw_model_output': raw_output,
            'parsed_tool_calls': calls
        }

        # 沒有 tool call，代表模型直接回答或已根據 tool result 進行回覆
        if not calls:
            answer = parsed_output['content'] or clean_generated_text(raw_output)

            if (
                messages[-1]['role'] == 'assistant'
                and 'tool_responses' in messages[-1]
                and 'content' not in messages[-1]
            ):
                messages[-1]['content'] = answer
            else:
                messages.append(
                    {
                        'role': 'assistant',
                        'content': answer
                    }
                )

            trace_item['final_answer'] = answer
            trace.append(trace_item)

            return answer, messages, trace

        # 模型要求執行工具
        tool_calls_for_history = []
        tool_responses = []

        for call in calls:
            tool_response = excute_tool_call(call)
            tool_responses.append(tool_response)

            tool_calls_for_history.append(
                {
                    'function':{
                        'name':call.get('name'),
                        'arguments': call.get('arguments',{})
                    }
                }
            )

        messages.append(
            {
            'role': 'assistant',
            'tool_calls': tool_calls_for_history,
            'tool_responses': tool_responses
                }
        )

        trace_item['tool_responses'] = tool_responses
        trace.append(trace_item)

    # 回到迴圈讓 generate_gemma 根據工具執行結果回答
    failure_message = '模型呼叫已達上限，無法完成這次請求。'

    messages.append(
        {
            'role': 'assistant',
            'content': failure_message
        }
    )

    return failure_message, messages, trace

In [19]:
answer, conversation, trace = run_agent('台北現在天氣如何?')

print('最終回答:')
print(answer)

最終回答:
台北目前的天氣是：

*   **溫度：** 25.3°C
*   **體感溫度：** 31.6°C
*   **天氣描述：** 輕微毛毛雨
*   **雲量：** 100%
*   **相對濕度：** 96%
*   **降雨量：** 0.1 mm
*   **風速：** 1.6 km/h


In [20]:
from pprint import pprint

pprint(trace)

[{'parsed_tool_calls': [{'arguments': {'location': 'Taipei'},
                         'name': 'get_current_weather'}],
  'raw_model_output': '<|tool_call>call:get_current_weather{location:<|"|>Taipei<|"|>}<tool_call|><|tool_response>',
  'round': 0,
  'tool_responses': [{'name': 'get_current_weather',
                      'response': {'data': {'apparent_temperature': 31.6,
                                            'cloud_cover_percent': 100,
                                            'data_source': 'Open-Meteo',
                                            'latitude': 25.05306,
                                            'longitude': 121.52639,
                                            'observation_time': '2026-08-26T00:00',
                                            'precipitation_mm': 0.1,
                                            'relative_humidity_percent': 96,
                                            'resolved_location': '台北市，臺灣省 or '
                                  

In [21]:
answer, _, trace = run_agent('請解釋 function calling 是什麼?')

print('最終回答:')
print(answer)

最終回答:
Function calling（函式呼叫）是一種在大型語言模型（LLM）中非常強大且實用的功能，它允許語言模型**理解用戶的意圖，並決定在需要時呼叫外部的、特定的程式碼或工具來執行任務**。

簡單來說，它讓語言模型從一個「純粹的文本生成器」轉變為一個「能夠執行動作的智能代理」。

以下我將從幾個面向來詳細解釋 Function Calling 的概念、運作方式以及它帶來的優勢：

### 1. Function Calling 的核心概念

Function Calling 的核心思想是建立一個**「工具/函式描述列表」**，將外部可用的功能（例如：查詢天氣、預訂訂票、計算數學、搜尋資料庫等）以結構化的方式提供給語言模型。

**運作流程可以概括為以下步驟：**

1. **定義工具 (Tool Definition)：** 開發者會向語言模型提供一系列可用的函式（例如：`get_current_weather(location)`、`search_database(query)`），並清楚說明每個函式的**名稱、用途（描述）以及所需的參數**。
2. **用戶輸入 (User Input)：** 用戶提出一個自然語言的請求，例如：「明天台北的天氣如何？」
3. **模型判斷 (Model Decision)：** 語言模型分析用戶的請求，並根據它所學到的知識和提供的工具描述，判斷**是否需要使用外部工具**來完成這個請求。
4. **生成呼叫 (Generate Call)：** 如果模型判斷需要使用工具，它不會直接生成答案，而是會輸出一個**結構化的指令（通常是 JSON 格式）**，明確指出要呼叫哪個函式以及傳入哪些參數。
    * *範例輸出：* `{"function_name": "get_current_weather", "parameters": {"location": "Taipei"}}`
5. **執行工具 (Tool Execution)：** 系統（應用程式）接收到模型的呼叫指令後，會**實際執行**對應的程式碼或 API 呼叫（例如，呼叫我系統中的 `get_current_weather` 工具）。
6. **回傳結果 (Return Result)：** 工具執行完畢後，會將**執行結果**（例如：`{"temp

In [22]:
answer, _, trace = run_agent('現在天氣如何？')

print(answer)
print(trace)

請問您想查詢哪個城市的天氣呢？請告訴我地點，我才能為您查詢。
[{'round': 0, 'raw_model_output': '請問您想查詢哪個城市的天氣呢？請告訴我地點，我才能為您查詢。<turn|>', 'parsed_tool_calls': [], 'final_answer': '請問您想查詢哪個城市的天氣呢？請告訴我地點，我才能為您查詢。'}]


In [23]:
conversation = None
SHOW_TOOL_TRACE = True

print("=" * 60)
print("Gemma 4 Tool Calling Agent")
print("=" * 60)
print()
print("指令：")
print("  exit     結束對話")
print("  clear    清除對話歷史")
print()

while True:
    user_message = input('\nYou:').strip()

    if user_message.lower() in {'exit', 'quit', '離開'}:
        print('離開對話')
        break

    if user_message.lower() == 'clear':
        conversation = None
        print('[System] 對話歷史已清除')
        continue

    if not user_message:
        continue
    try:
        answer, conversation, trace = run_agent(
            user_message = user_message,
            conversation = conversation
        )

        if SHOW_TOOL_TRACE:
            for item in trace:
                tool_calls = item.get('parsed_tool_calls', [])
                tool_responses = item.get('tool_responses', [])

                if tool_calls:
                    print('\n[Tool Call]')
                    for call in tool_calls:
                        print(f'function: {call['name']}')
                        print(f'arguments: {call['arguments']}')

                if tool_responses:
                    print('\n[Tool Response]')
                    for response in tool_responses:
                        print(response)

        print('\nGemma:')
        print(answer)
    except Exception as e:
        print('\n[Error]')
        print(f'{type(e).__name__()}: {e}')


Gemma 4 Tool Calling Agent

指令：
  exit     結束對話
  clear    清除對話歷史


You:請介紹你自己

Gemma:
您好，我是一個大型語言模型，由 Google 訓練。

我的主要功能是理解和生成自然語言，這使我能夠：

*   **回答問題：** 根據我所擁有的龐大知識庫，提供各種主題的資訊和解釋。
*   **生成文本：** 撰寫文章、摘要、故事、程式碼、信件等各種風格的文字。
*   **翻譯與總結：** 協助進行語言翻譯，並將長篇文本精簡為重點摘要。
*   **進行推理與創意發想：** 協助您進行腦力激盪、解決複雜問題，或提供創新的想法。
*   **執行特定任務：** 根據您提供的指令，例如使用工具來查詢即時天氣等。

**我的特點包括：**

1.  **專業性與準確性：** 我致力於提供準確、有條理的資訊。
2.  **多語言能力：** 我可以理解和生成多種語言的文本。
3.  **持續學習：** 我會不斷地從數據中學習和改進我的表現。

簡單來說，您可以把我視為一位知識豐富、高效且樂於助人的數位助理。

請問您現在需要我為您做些什麼呢？

You:我想知道板橋的天氣

[Tool Call]
function: get_current_weather
arguments: {'location': 'Banqiao'}

[Tool Response]
{'name': 'get_current_weather', 'response': {'ok': True, 'data': {'resolved_location': '板橋區，臺北市，台湾', 'latitude': 25.01427, 'longitude': 121.46719, 'timezone': 'Asia/Taipei', 'observation_time': '2026-08-26T00:00', 'temperature': 25.7, 'temperature_unit': '°C', 'apparent_temperature': 31.4, 'relative_humidity_percent': 91, 'precipitation_mm': 0.0, 'cloud_cover_percent': 100,